# Maximal visits: demo

Concept: `overview/Single Level Visit Mappings.md` (outside this repo), sections "Visits" (with DD-V1 to DD-V3), "Properties", "Deriving maximal Visits", "Visit duration" and "implementation".
Code: modules `ocelescope_module_location_based_ocpm.visits.order`, `.context`, `.maximal`, `.pipeline` and `.visit_files`; this notebook only imports and calls their functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(add_gaps)`) and `docs/resources/visits.md`.
Stops, transfers and shifts of the visits: `shifts.ipynb`; waiting times (with or without visits): `waiting_times.ipynb`.

What the notebook does:
1. loads the running example (Level 3) and the small edge-case example, with their location context,
2. derives the maximal visits step by step, with the options of DD-V1 and DD-V2,
3. shows the flag for naming clashes of the auto-completed location mapping,
4. runs everything with one call (`analyse_visits`),
5. goes through the edge cases of visits on the small example,
6. saves the maximal visits of the running example to `data/running_example/derived/`.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/visits.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from datetime import timedelta
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA, load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import add_high_level_locations, location_context, naming_clashes
from ocelescope_module_location_based_ocpm.visits.maximal import (
    add_gap_measures,
    add_gaps,
    add_kappa,
    add_trace_positions,
    add_visit_ids,
    derive_maximal_visits,
    maximal_visits,
    visit_relations,
)
from ocelescope_module_location_based_ocpm.visits.pipeline import analyse_visits
from ocelescope_module_location_based_ocpm.visits.trace_figure import relations_for_figure, trace_figure
from ocelescope_module_location_based_ocpm.visits.visit_files import load_visits, save_visits

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"
EDGE = ROOT / "data" / "visits_example"

# Show whole frames.
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Inputs

### Running example

The location mapping of Level 3, as saved by `location_derivation.ipynb` (`load_level_mapping(folder, level)`), and the location context from `data/running_example/inputs/locations.csv` (`location_context(table)`, see `trace_figure.ipynb`, section 1): $agg$ maps A1, A2 → A and B1 → B; T is not mapped, so it is its own high-level location; $V = \{T\}$, storage units $W = \{B1\}$.

The excursion tolerance is $\kappa_{max} = 2$, as in the note's figure of the maximal visits.

In [ ]:
LEVEL, KAPPA = 3, 2
mapping = load_level_mapping(DERIVED, LEVEL)
context = location_context(pl.read_csv(EXAMPLE / "inputs" / "locations.csv"))
context

### Small edge-case example

`data/visits_example/` holds a small example made for these notebooks (generated by `make_visits_example.py`, the single source of its files): eight handling units h1 to h8 whose traces cover the edge cases.
The relations are in the format of a saved level mapping (read with the same schema `MAPPING_SCHEMA`); `loc` is given directly, as if derived.

Location context: A1, A2, the **crane C1** (a vehicle mapped *into* A) and the **storage units S1, S2** lie in hall A, B1 and the storage unit B2 in hall B, and the **truck T** is not mapped (a vehicle with its own high-level location); $V = \{C1, T\}$, $W = \{S1, S2, B2\}$.
Processing operations $P$ are given explicitly in `processing_events.csv`; three of them are movements ($P \cap M \neq \emptyset$).
The example is meant for $\kappa_{max} = 1$.

| handling unit | edge cases of visits (section 5) |
|---|---|
| h1 | T visited twice (cut by κ); with `ignore_vehicles` and κ = 0 the visit to A runs on across a truck trip |
| h3 | ⊥ relations (as at Level 0): two ⊥ visits because $\kappa(\bot) = 0$; ⊥ inside a visit to A |
| h4 | a high-level location visited twice (cut by κ) |
| h7 | events with the same timestamp, ordered by AS4 against their ids |

The other handling units are about shifts (`shifts.ipynb`) and waiting times (`waiting_times.ipynb`).

In [ ]:
EDGE_KAPPA = 1
edge_relations = pl.read_csv(EDGE / "relations.csv", schema=MAPPING_SCHEMA)
edge_context = location_context(pl.read_csv(EDGE / "locations.csv"))
edge_processing = pl.read_csv(EDGE / "processing_events.csv")["event"].to_list()
print(edge_context)
print("P =", edge_processing)
edge_relations

## 2. Maximal visits, step by step

Concept: a visit $(hu, hl, \sigma)$ is an infix $\sigma = otrace(hu)[i:k]$ that enters $hl$ at $e_i$, ends with the last event at $hl$ at $e_k$, and contains no deviating infix longer than $\kappa(hl)$ ($\kappa(\bot) = 0$).
The maximal visits are found as in the note's "Deriving maximal Visits": take the positions of $hu$ at $hl$, cut wherever the gap between two consecutive ones is longer than $\kappa(hl)$.
Each step below adds one or more columns to the relations frame; shown for c1, the note's example.

### Trace positions: `add_trace_positions(relations)`

Sorts the relations in $seq$ order per handling unit and adds `position`, the index $j$ of the relation in $otrace(hu) = \langle e_1, \dots, e_n \rangle$, starting at 1.
Events of one handling unit with the same timestamp are ordered by **AS4**: movement events first, then processing operations, then all other events, each group by event id (an event that is both a movement and a processing operation counts as a movement).
The group is the column `tie_rank` (0, 1, 2; `order.tie_rank`); the running example has no equal timestamps, h7 of the small example shows the effect (section 5.4). The same order is used for shifts and waiting times.

| parameter | meaning | default |
|---|---|---|
| `relations` | one row per hu-event relation with columns hu, time, event, is_movement (and is_transformation if `processing` is `None`) | |
| `processing` | $P$ as event ids, for the tie-break; use the same $P$ as for the waiting times | `None` = $T$ |
| `movement` | column marking the movement events $M$ | `"is_movement"` |

In [ ]:
rel = add_trace_positions(mapping)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "time", "is_movement", "is_transformation", "loc", "tie_rank", "position")

### High-level locations: `add_high_level_locations(relations, location, agg, high_level="high_level")`

$\widehat{hl}((e,hu)) = agg(loc((e,hu)))$ if $loc((e,hu)) \neq \bot$, else $\bot$; $agg$ may be partial and is completed by the identity (here: T ↦ T).

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `location` | |
| `location` | column of $loc$ | |
| `agg` | dict low-level → high-level location | identity |
| `high_level` | name of the new column | `"high_level"` |

In [ ]:
rel = add_high_level_locations(rel, "loc", context.agg)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "loc", "position", "high_level")

### Tolerance: `add_kappa(relations, kappa, high_level="high_level")`

Adds `kappa` $= \kappa(\widehat{hl})$, the tolerance of the relation's own high-level location: $\kappa_{max}$ for every $hl \in HL$, 0 for ⊥.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `high_level` | |
| `kappa` | $\kappa_{max} \geq 0$ | |
| `high_level` | column of $\widehat{hl}$ | `"high_level"` |

The concept has one $\kappa_{max}$; a tolerance per high-level location would only change this function.

In [ ]:
rel = add_kappa(rel, KAPPA)
rel.filter(pl.col("hu").is_in(["c1", "c5"])).select("hu", "event", "high_level", "kappa")

### Gaps: `add_gaps(relations, high_level="high_level", location="loc", vehicles=(), ignore_vehicles=False)`

Adds `gap` $g_j = p_j - p_{j-1} - 1$: the number of events since the previous relation of the same handling unit at the same high-level location (⊥ counts as one location here); null for the first one.

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `add_trace_positions` with the column `high_level` | |
| `high_level` | column of $\widehat{hl}$ | `"high_level"` |
| `location`, `vehicles` | column of $loc$ and the set $V$; only used with `ignore_vehicles` | `"loc"`, empty |
| `ignore_vehicles` | `True`: relations on a vehicle ($loc \in V$) in between do not count towards the gap, even if the vehicle maps to another high-level location (DD-V2, see the end of this section); gaps before ⊥ relations always count every event | `False` |

How it works: every relation gets a weight $w$ (1, or 0 for a relation on a vehicle with `ignore_vehicles`); with $C$ the cumulative sum of $w$ per handling unit, the gap is $C(p_j) - w(p_j) - C(p_{j-1})$, which is $p_j - p_{j-1} - 1$ when all weights are 1. `over("hu", "high_level")` puts all ⊥ of a handling unit into one group, so ⊥ needs no stand-in value.

Example (note, table for κ = 2): c1 at A has positions 1, 2, 3, 6, 7 and gaps –, 0, 0, 2, 0.

In [ ]:
rel = add_gaps(rel)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "position", "high_level", "gap")

### Further measures of the deviating infixes: `add_gap_measures(relations, high_level="high_level")`

DD-V1: $\kappa$ counts events, but other limits can be selected too, alone or combined (all selected conditions must hold).
For this, every relation gets two more measures of the deviating infix before it, $otrace(hu)[p_{j-1} + 1 : p_j - 1]$ (all of its events are not at $hl$):

| column | meaning |
|---|---|
| `gap_timestamps` | number of distinct timestamps in the infix (0 if it is empty) |
| `gap_duration` | $time(e_{p_j}) - time(e_{p_{j-1}+1})$: from the first event of the infix until the first event back at $hl$ (0 if it is empty) |

Both are null for the first relation of a handling unit at a high-level location. Relations on vehicles always count here (`ignore_vehicles` only concerns the number of events).

How it works: the timestamps of a trace get a dense rank $t$ (the trace is sorted by time), so the distinct timestamps of positions $a, \dots, b$ are $t(b) - t(a) + 1$; the values of the relation after $p_{j-1}$ are taken with `shift(-1)` per handling unit and then `shift(1)` per (hu, hl).

Example: c1's relation e9 at A follows the infix ⟨e7 at B1, e8 on T⟩: gap 2, 2 timestamps, $time(e9) - time(e7) = 2$ h.

In [ ]:
rel = add_gap_measures(rel)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "time", "position", "high_level", "gap", "gap_timestamps", "gap_duration")

### Visit ids: `add_visit_ids(relations, high_level="high_level", max_timestamps=None, max_excursion_duration=None)`

Adds `starts_visit` and `visit_id`, the maximal visit at the relation's own high-level location.
A relation starts a new maximal visit iff it is the first one of its handling unit at its high-level location (`gap` null) or the deviating infix before it violates one of the selected conditions (DD-V1):

| condition | parameter | default |
|---|---|---|
| more counted events than $\kappa$ | `kappa` (column, from `add_kappa`; `kappa=None` there = no limit) | κ |
| more distinct timestamps than `max_timestamps` | `max_timestamps` | `None` (off) |
| longer than `max_excursion_duration` | `max_excursion_duration` | `None` (off) |

At ⊥ the tolerance is always 0 (DD-V3): any relation in between cuts, whatever is selected.
The ids are numbered 1, 2, … over all visits, ordered by handling unit and first position.

How it works: a cumulative sum of `starts_visit` per (hu, high_level) numbers the visits; the visit's first position and the handling unit give the id (dense rank).

Example: with κ = 2, no gap of c1 is longer than 2, so c1 has one visit per high-level location (A, B, T).

In [ ]:
rel = add_visit_ids(rel)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "position", "high_level", "gap", "kappa", "starts_visit", "visit_id")

### The visits: `maximal_visits(relations, high_level="high_level")`

One row per maximal visit $\nu = (hu, hl, otrace(hu)[i:k])$:

| column | meaning |
|---|---|
| `visit_id`, `hu`, `high_level`, `kappa` | the visit (high_level null = ⊥ visit) and its tolerance |
| `first_position`, `last_position` | $i$ and $k$ |
| `first_event`, `last_event` | $e_i$ (= the entry event $e_{in}$) and $e_k$ |
| `exit_event` | $e_{out} = otrace(hu)[k+1]$, or $e_k$ if the trace ends with the visit |
| `ends_trace` | true if the trace ends with the visit (then the duration is a lower bound) |
| `entry_time`, `last_time`, `exit_time` | times of $e_{in}$, $e_k$, $e_{out}$ |
| `duration` | $d = time(e_{out}) - time(e_{in})$ |
| `n_relations`, `n_at_location` | $\lvert\sigma\rvert$ and the number of relations at $hl$ |
| `interrupted` | true iff some relation of the visit is not at $hl$ |

How it works: group by `visit_id` (first/last position, event and time); the exit event is the relation of the same handling unit at `last_position + 1` (a join), if there is one.

These are exactly $m_1$ to $m_{11}$ of the note (κ = 2); the ids differ only in their order (the note starts with s1): visit 1 = $m_2$, 2 = $m_3$, 3 = $m_4$, 4 = $m_5$, 5 = $m_6$, 6 = $m_7$, 7 = $m_8$, 8 = $m_9$, 9 = $m_{10}$, 10 = $m_{11}$ (c5 at ⊥), 11 = $m_1$.

In [ ]:
visits = maximal_visits(rel)
visits

### Relations of the visits: `visit_relations(relations, visits, high_level="high_level")`

One row per visit and relation $r \in R(\nu)$, i.e. every relation of the visit's infix, including those of its excursions.
A relation therefore appears once for its own maximal visit and once for every visit at another location around it (e.g. e8 of c1 is in visit 3 = $m_4$ at T and, as an excursion, in visit 1 = $m_2$ at A).

| column added | meaning |
|---|---|
| `visit_id`, `visit_high_level`, `first_position`, `last_position` | the visit the row belongs to |
| `at_location` | $\widehat{hl}(r) = hl$ (for a ⊥ visit: $\widehat{hl}(r) = \bot$) |

How it works: each visit is expanded to its positions $i, \dots, k$ (`int_ranges` + `explode`) and joined with the relations on (hu, position); the cost is linear in the size of the result, no pairwise comparison of visits and relations.
It works for any set of visits given by hu, first and last position, not only maximal ones.

In [ ]:
vrel = visit_relations(rel, visits)
vrel.filter(pl.col("hu") == "c1").select("visit_id", "visit_high_level", "hu", "event", "position", "loc", "high_level", "at_location")

### All in one call: `derive_maximal_visits(relations, location="loc", agg=None, kappa=0, vehicles=(), ignore_vehicles=False, processing=None, max_timestamps=None, max_excursion_duration=None)`

Runs the steps above (trace positions, high-level locations, κ, gaps and gap measures, visit ids, visits) and returns the relations and the visits.

| parameter | meaning | default |
|---|---|---|
| `relations` | e.g. a loaded level mapping or `Derivation.relations` | |
| `location` | column of $loc$ (e.g. `"loc_3"` on `Derivation.relations`) | `"loc"` |
| `agg` | location mapping | identity |
| `kappa` | $\kappa_{max}$, the maximum number of events of a deviating infix; `None` = no limit (DD-V1) | 0 |
| `vehicles`, `ignore_vehicles` | see `add_gaps` (DD-V2) | empty, `False` |
| `processing` | $P$, only for the order of events with the same timestamp (see `add_trace_positions`) | `None` = $T$ |
| `max_timestamps`, `max_excursion_duration` | further conditions of DD-V1 (see `add_visit_ids`) | `None`, `None` |

Below: the same visits again, and the number of maximal visits for κ = 0, 1, 2 (review 1: 13, 12, 10 at high-level locations, plus the ⊥ visit of c5).

In [ ]:
rel_k, visits_k = derive_maximal_visits(mapping, "loc", context.agg, KAPPA)
print("same as step by step:", visits_k.equals(visits))
{kappa: derive_maximal_visits(mapping, "loc", context.agg, kappa)[1].height for kappa in (0, 1, 2)}

### Option (DD-V2): vehicles do not use up the tolerance (`ignore_vehicles=True`)

By default, every event in a deviating infix counts towards $\kappa$, also relations on a vehicle that maps to another high-level location (here the truck T).
With `ignore_vehicles=True` (in `add_gaps`, `derive_maximal_visits` and `analyse_visits`; default `False`), relations on a vehicle do not count: a trip on the truck between two stays at A does not use up A's tolerance.
Visits to ⊥ are not affected (they allow no excursions at all).

Example with κ = 1: c1's relations between e5 and e9 are ⟨e7 at B1, e8 on T⟩. By default the gap is 2 > 1 and the visit to A is cut into e3–e5 and e9–e11; with `ignore_vehicles=True` only B1 counts, the gap is 1, and the visit to A stays e3–e11. T itself is still cut (e8 and e13–e14), since A2 is not a vehicle.

In [ ]:
for ignore in (False, True):
    rel_1, visits_1 = derive_maximal_visits(mapping, "loc", context.agg, 1, context.vehicles, ignore_vehicles=ignore)
    print(f"ignore_vehicles={ignore}: {visits_1.height} maximal visits")
    print(visits_1.filter(pl.col("hu") == "c1").select("visit_id", "high_level", "first_event", "last_event"))

### Options (DD-V1): limits on timestamps and on the duration of an excursion

c1's visit to A contains the deviating infix ⟨e7 at B1, e8 on T⟩: 2 events, 2 timestamps, 2 h from e7 until e9 (back at A).
With `kappa=None` (no limit on the number of events) and
- no further condition: A stays e3–e11;
- `max_timestamps=1`: 2 timestamps > 1, so A is cut into e3–e5 and e9–e11;
- `max_excursion_duration=timedelta(hours=1)`: 2 h > 1 h, cut as well; with 2 h it is not cut.
Conditions combine: with `kappa=2` (holds) and a limit of 1 h (violated), A is cut.

In [ ]:
options = {
    "kappa=None": dict(kappa=None),
    "kappa=None, max_timestamps=1": dict(kappa=None, max_timestamps=1),
    "kappa=None, max_excursion_duration=1 h": dict(kappa=None, max_excursion_duration=timedelta(hours=1)),
    "kappa=None, max_excursion_duration=2 h": dict(kappa=None, max_excursion_duration=timedelta(hours=2)),
    "kappa=2, max_excursion_duration=1 h": dict(kappa=2, max_excursion_duration=timedelta(hours=1)),
}
for label, kwargs in options.items():
    _, visits_option = derive_maximal_visits(mapping, "loc", context.agg, **kwargs)
    c1_a = visits_option.filter((pl.col("hu") == "c1") & (pl.col("high_level") == "A"))
    print(f"{label}: {c1_a.select('first_event', 'last_event').rows()}")

The maximal visits in the trace figure (see `trace_figure.ipynb`): the figure only needs a `group` column next to the visit columns.

In [ ]:
HUS = ["s1", "c1", "c2", "c3", "c4", "c5"]
figure_relations = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units)
trace_figure(
    figure_relations,
    visits.with_columns(pl.lit("maximal").alias("group")),
    title=f"Running example, Level {LEVEL}: maximal visits at κ = {KAPPA}",
).show()

## 3. Flag: naming clashes of the auto-completed mapping: `naming_clashes(locations, agg)`

$agg$ may be partial; unmapped locations are their own high-level location (identity).
If an unmapped low-level location has the same name as a high-level location that other locations are mapped to (e.g. B unmapped and B1 → B), both end up at B, one intentionally and one not ("implementation" section of the note). This is flagged, not a blocking error.

| parameter | meaning |
|---|---|
| `locations` | all low-level locations that occur |
| `agg` | the provided mapping |

Returns one row per clash: the unmapped `location` and the `mapped_locations` mapped to the same name. The running example has none; a made-up location list with an unmapped B shows one.

In [ ]:
print(naming_clashes(mapping["loc"].drop_nulls().unique().to_list(), context.agg))
naming_clashes(["A1", "A2", "B", "B1", "T"], context.agg)

## 4. All in one call: `analyse_visits(relations, location="loc", agg=None, vehicles=(), kappa=0, processing=None, ignore_vehicles=False, max_timestamps=None, max_excursion_duration=None, vehicles_outside_external=False)`

Runs section 2, the shifts of `shifts.ipynb`, and the flag of section 3.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame (eager or lazy) | |
| `location`, `agg`, `kappa`, `vehicles`, `processing`, `ignore_vehicles`, `max_timestamps`, `max_excursion_duration` | as for `derive_maximal_visits` | |
| `vehicles_outside_external` | DD-S1, see `shifts.ipynb` | `False` |

Returns a `VisitAnalysis` with the (collected) frames `relations`, `visits` (visit and shift columns), `visit_relations`, `transfers` and `naming_clashes`.
Waiting times are computed separately (`waiting_times.ipynb`), with these visits or without visits.

In [ ]:
analysis = analyse_visits(mapping, "loc", context.agg, context.vehicles, kappa=KAPPA)
print("same visits as step by step:", analysis.visits.select(visits.columns).equals(visits))
analysis.visits.select(visits.columns)

## 5. Edge cases on the small example

The small example with $\kappa_{max} = 1$ and the explicit $P$.
The figure uses the trace position axis, since the traces share no events.

In [ ]:
edge = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
)
edge_figure_relations = relations_for_figure(
    edge_relations, "loc", ["h1", "h2", "h3", "h4", "h5", "h6", "h7", "h8"], edge_context.agg, edge_context.vehicles,
    edge_context.storage_units, axis="trace position", processing=edge_processing,
)
trace_figure(
    edge_figure_relations,
    edge.visits.with_columns(pl.lit("maximal").alias("group")),
    title=f"Edge-case example: maximal visits at κ = {EDGE_KAPPA}",
).show()
edge.visits.select(visits.columns).drop("kappa", "first_position", "last_position", "entry_time", "last_time", "exit_time")

### 5.1 ⊥ (h3)

h3: ⊥, ⊥, A1, ⊥, A1, T, A2 (unknown locations in between, as at Level 0; from Level 1 on, ⊥ only occurs at the start of a trace).
- $\kappa(\bot) = 0$ (DD-V3): e19–e20 is one ⊥ visit, e22 is a second one (gap 1 > 0).
- The visit to A (e21–e25) contains e22 at ⊥ as an excursion; it is interrupted.

In [ ]:
edge.visits.filter(pl.col("hu") == "h3").select(
    "visit_id", "high_level", "first_event", "last_event", "exit_event", "n_relations", "n_at_location", "interrupted",
)

### 5.2 Cuts by κ (h4, h1)

- h4 visits A at e27 and at e32 with a gap of 4 > κ: **two maximal visits** to A. The same happens to h1 at T (e6 and e11).

In [ ]:
edge.visits.filter(pl.col("hu").is_in(["h4", "h1"]) & pl.col("high_level").is_in(["T", "A"])).select(
    "visit_id", "hu", "high_level", "first_event", "last_event", "n_relations",
)

### 5.3 Vehicles and the tolerance (h1)

With `ignore_vehicles=True` and κ = 0, h1's visit to A runs on across ⟨e6 on T⟩ (the truck trip uses no tolerance) and is only cut at e8 (B1); by default it is cut at e6 as well.

In [ ]:
for ignore in (False, True):
    _, visits_0 = derive_maximal_visits(edge_relations, "loc", edge_context.agg, 0, edge_context.vehicles, ignore_vehicles=ignore)
    print(f"ignore_vehicles={ignore}")
    print(visits_0.filter((pl.col("hu") == "h1") & (pl.col("high_level") == "A")).select("visit_id", "first_event", "last_event"))

### 5.4 Events with the same timestamp (h7)

AS4 orders events with the same timestamp: movements, then processing operations, then the rest, each group by event id; the ids of h7 are chosen against this order.
At t = 3 the crane move e55 comes before the grind e54, at t = 5 the roll e57 (∈ P) before the label e56.
With AS4, the grind at t = 3 is at A1 like the move, as RQ4 requires. The effect on the waiting times is in `waiting_times.ipynb`.

In [ ]:
edge.relations.filter(pl.col("hu") == "h7").select("event", "time", "activity", "is_movement", "loc", "tie_rank", "position")

## 6. Save the maximal visits: `save_visits(visits, folder, level, kappa)` and `load_visits(folder, level, kappa)`

Writes the core columns of the maximal visits (those of `maximal_visits` without the duration; CSV has no duration or list type) to `data/running_example/derived/maximal_visits_level_{k}_kappa_{κ}.csv`, where later notebooks load them from; `load_visits` reads them and recomputes the duration.

In [ ]:
path = save_visits(analysis.visits, DERIVED, LEVEL, KAPPA)
print(path.relative_to(ROOT))
load_visits(DERIVED, LEVEL, KAPPA)